# Statistical Analysis

## Project Objective

The purpose of this notebook is to investigate selected relationships identified during exploratory data analysis using statistical methods.

While EDA helped us identify patterns and differences in the dataset, statistical analysis will help us evaluate whether selected differences are supported by statistical evidence.

## Analysis Approach

For each investigation, we will:

1. Define a research question.
2. Formulate the null and alternative hypotheses.
3. Select an appropriate statistical method.
4. Check the assumptions of the selected method.
5. Perform the statistical analysis.
6. Interpret statistical significance and effect size.
7. Discuss the practical implications for fraud detection.

## First Investigation: CVV Retry Count and Fraud

**Research question:** Does the fraud rate differ between transactions with zero CVV retries and transactions with at least one CVV retry?

We will compare the proportion of fraudulent transactions between these two groups.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from scipy.stats import chi2_contingency


In [6]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"

# Load data
train_path = RAW_DATA_DIR / "train.csv"
train_df = pd.read_csv(train_path)

print(f"Training data shape: {train_df.shape}")

Training data shape: (699960, 26)


#### STEP 1: Create our two groups

In [7]:
# Create a separate DataFrame for this investigation
cvv_analysis_df = train_df[["cvv_retry_count", "is_fraud"]].copy()

# Group transactions by CVV retries
cvv_analysis_df["cvv_retry_group"] = (
    cvv_analysis_df["cvv_retry_count"]
    .gt(0)
    .map({
        False: "Zero retries",
        True: "One or more retries"
    })
)

# Summarise each group
cvv_summary = (
    cvv_analysis_df
    .groupby("cvv_retry_group")
    .agg(
        total_transactions=("is_fraud", "count"),
        fraudulent_transactions=("is_fraud", "sum"),
        fraud_rate=("is_fraud", "mean")
    )
)

# Convert fraud rate to percentage
cvv_summary["fraud_rate"] *= 100

cvv_summary

,total_transactions,fraudulent_transactions,fraud_rate
cvv_retry_group,,,
One or more retries,57872,5699.0,9.847595
Zero retries,642088,5990.0,0.932894


In [8]:
contingency_table = pd.crosstab(
    cvv_analysis_df["cvv_retry_group"],
    cvv_analysis_df["is_fraud"]
)

# Rename outcome columns
contingency_table.columns = ["Legitimate", "Fraudulent"]

contingency_table

,Legitimate,Fraudulent
cvv_retry_group,,
One or more retries,52173,5699
Zero retries,636098,5990


In [10]:

chi2_stat, p_value, degrees_of_freedom, expected_counts = (
    chi2_contingency(contingency_table)
)

print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"P-value: {p_value:.10f}")
print(f"Degrees of freedom: {degrees_of_freedom}")

print("\nExpected counts:")
print(
    pd.DataFrame(
        expected_counts,
        index=contingency_table.index,
        columns=contingency_table.columns
    )
)

Chi-square statistic: 25687.4625
P-value: 0.0000000000
Degrees of freedom: 1

Expected counts:
                        Legitimate    Fraudulent
cvv_retry_group                                 
One or more retries   56905.565049    966.434951
Zero retries         631365.434951  10722.565049


#### Chi-square Test Results

The Chi-square test of independence was used to investigate whether CVV retry group and fraud outcome are associated.

**Results:**
- Chi-square statistic: 25,687.4625
- Degrees of freedom: 1
- P-value: Less than 0.0000000001

All expected cell counts were greater than 5, satisfying the expected-frequency requirement.

**Decision:** Reject the null hypothesis at the 5% significance level.

**Interpretation:**

There is strong statistical evidence of an association between CVV retry group and fraud outcome.

Transactions with at least one CVV retry had a substantially higher observed fraud rate (9.85%) compared with transactions with zero retries (0.93%).

This finding is an association and does not establish causation.

The result suggests that CVV retry behaviour may be a useful signal to investigate during fraud prediction modelling.

In [12]:
# Calculate Cramér's V
n = contingency_table.to_numpy().sum()

minimum_dimension = min(contingency_table.shape) - 1

cramers_v = np.sqrt(
    chi2_stat / (n * minimum_dimension)
)

print(f"Cramér's V: {cramers_v:.4f}")

Cramér's V: 0.1916


In [13]:
# Extract fraud rates
retry_fraud_rate = (
    contingency_table.loc["One or more retries", "Fraudulent"]
    / contingency_table.loc["One or more retries"].sum()
)

zero_retry_fraud_rate = (
    contingency_table.loc["Zero retries", "Fraudulent"]
    / contingency_table.loc["Zero retries"].sum()
)

# Calculate risk ratio
risk_ratio = retry_fraud_rate / zero_retry_fraud_rate

print(f"Risk Ratio: {risk_ratio:.4f}")

Risk Ratio: 10.5560
